# Epidemic allocation research
Upload research_source.zip from this project's colab folder. Select a GPU runtime to use optional PyTorch CUDA lookahead; auto backend falls back to NumPy on CPU. Run parity tests before experiments. Synthetic validation is explicitly labeled. Prepare empirical data separately and record provenance. Colab execution and account login are performed by you.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
files.upload()
with zipfile.ZipFile('research_source.zip') as bundle:
    bundle.extractall('/content/epidemic_research')
%cd /content/epidemic_research
%pip install -q -r requirements.txt

In [ ]:
!python -m unittest discover -s tests -v
!python -m epidemic.benchmark --counties 3126 --horizon 28 --backend auto
import json
config=json.load(open('config.example.json'))
config['rollout_backend']='auto'
config['stochastic']=True
json.dump(config,open('colab_config.json','w'),indent=2)

## Synthetic debugging example
These outcomes are not empirical evidence. Use several seeds only with stochastic execution; deterministic seeds repeat the same outcome.

In [ ]:
!python -m epidemic.experiments --synthetic --config colab_config.json --counties 24 --days 28 --seeds 3 --output outputs/demo
from IPython.display import Image, display
display(Image('outputs/demo/infections.png'))

## Empirical inputs
Upload a prepared_data.zip with counties.csv and mobility.csv at its root, after following README.md. CTIS training targets and TBCOV features must be temporally aligned. The CTIS mask signal starts September 8, 2020, so July 2020 decisions using CTIS training are retrospective. Intervention cost assumptions must be explicit.

In [ ]:
# Uncomment after preparing and uploading the empirical bundle.
# files.upload()
# with zipfile.ZipFile('prepared_data.zip') as bundle:
#     bundle.extractall('data/prepared')
# !python -m epidemic.experiments --data data/prepared --config config.example.json --days 84 --seeds 20 --output outputs/empirical

## TBCOV streaming
Use sentiment_label and resident-location fields from base TSV releases; hydration is unnecessary for these labels. Monthly archives are large. Download only selected months to sufficiently large storage, then aggregate in chunks with epidemic.tbcov. Country/language ID lists alone do not contain the base labels. Do not treat place mentions in tweet text as residence.

In [ ]:
# !python -m epidemic.tbcov selected_month.tsv county_crosswalk.csv --output outputs/tbcov_county_month.csv
# !python -m epidemic.prediction features.csv feature_sets.json --folds 5
# !python -m epidemic.sensitivity --synthetic --days 28 --seeds 5

In [ ]:
import shutil
shutil.make_archive('/content/research_outputs','zip','outputs')
files.download('/content/research_outputs.zip')